In [24]:
import pandas as pd
import numpy as np
import yfinance as yf
import requests 
import os
import logging
import time
from tqdm import tqdm

In [ ]:
# Globals

raw_data_dir = "../data/raw"
proc_data_dir = "../data/processed"

# EDIT THIS!!!
headers = {
    "User-Agent": "First Last email@gmail.com"
}

In [7]:
# Setting up logger
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, filename = "log.log", filemode = "w", 
                    format = "%(asctime)s - %(levelname)s - %(message)s")

## Creating Datasets

In [11]:
# File downloader

def download_file(url: str, filename: str, headers: dict = {}) -> bool:
    save_path = os.path.join(raw_data_dir, filename)

    if os.path.exists(save_path):
        logger.info(f"{filename} already exists. Skipping download.")
        return True
    
    try:
        logger.info(f"Downloading {filename} from web...")

        response = requests.get(url, headers = headers, stream = True)

        response.raise_for_status()

        with open(save_path, 'wb') as file:
            for chunk in response.iter_content(chunk_size=8192):
                file.write(chunk)

        logger.info(f"Successfully saved to {save_path}")
        return True
    
    except Exception as e:
        logger.error(f"Failed to download {filename}: {e}")
        return False


In [14]:
download_file("https://www.sec.gov/files/company_tickers.json","company_tickers.json", headers)

True

In [16]:
df = pd.read_json("../data/raw/company_tickers.json", orient = "index")
df['cik_str'] = df['cik_str'].astype(str).str.zfill(10)

display(df.head())

,cik_str,ticker,title
0,0001045810,NVDA,NVIDIA CORP
1,0001652044,GOOGL,Alphabet Inc.
2,0000320193,AAPL,Apple Inc.
3,0000789019,MSFT,MICROSOFT CORP
4,0001018724,AMZN,AMAZON COM INC


In [ ]:
def fetch_single_comp_metrics(ticker: str) -> dict:

    try:
        stock = yf.Ticker(ticker)
        info = stock.info

        return {
            "ticker": ticker,
            
            # Quantitative block for ML
            "forwardPE": info.get("forwardPE"),
            "ev_to_ebitda": info.get("enterpriseToEbitda"),
            "ebitda_margin": info.get("ebitdaMargins"),
            "debt_to_equity": info.get("debtToEquity"),
            
            # Qualitative block for ML
            "sector": info.get("sector", "Unknown"),
            "industry": info.get("industry", "Unknown"),
            "business_summary": info.get("longBusinessSummary", ""),
            
            # Data for valuation
            "ebitda": info.get("ebitda"),
            "total_cash": info.get("totalCash"),
            "total_debt": info.get("totalDebt"),
            "shares_outstanding": info.get("sharesOutstanding")
        }
    except Exception as e:
            logger.warning(f"Failed to fetch data for {ticker}: {e}")
            return None



In [ ]:
# Testing function
fetch_single_comp_metrics("NVDA")

{'ticker': 'NVDA',
 'forwardPE': 17.73965,
 'ev_to_ebitda': 35.888,
 'ebitda_margin': 0.61698,
 'debt_to_equity': 7.255,
 'sector': 'Technology',
 'industry': 'Semiconductors',
 'business_summary': "NVIDIA Corporation operates as a data center scale AI infrastructure company. The company operates through two segments, Compute & Networking, and Graphics segments. The Compute & Networking segment provides data center accelerated computing and networking platforms and artificial intelligence solutions and software, and automotive platforms and autonomous and electric vehicle solutions, including software. The Graphics segment offers GeForce GPUs for gaming and PCs; Quadro/NVIDIA RTX GPUs for enterprise workstation graphics. The company's products are used in gaming, professional visualization, data center, and automotive markets. The company sells its products to original equipment manufacturers, original device manufacturers, system integrators and distributors, independent software vend

In [25]:
def build_csv_comps_table(raw_data_path: str, output_csv: str, chunk_size: int = 50, limit: int = None):
        logger.info("Starting large data pull...")
        df_raw = pd.read_json(raw_data_path, orient='index')
        all_tickers = df_raw['ticker'].tolist()

        if limit:
            all_tickers = all_tickers[:limit]
            logger.info(f"Test Mode: Only processing the first {limit} companies")
        
        # Check if a partial file already exists to resume
        start_index = 0
        if os.path.exists(output_csv):
            existing_df = pd.read_csv(output_csv)
            start_index = len(existing_df)
            logger.info(f"Found existing file. Resuming from ticker {start_index}...")

        for i in range(start_index, len(all_tickers), chunk_size):
            chunk = all_tickers[i : i + chunk_size]
            chunk_data = []
            
            for ticker in tqdm(chunk, desc=f"Chunk {i//chunk_size}", leave=False):
                metrics = fetch_single_comp_metrics(ticker)
                if metrics:
                    chunk_data.append(metrics)
            
            # Save the chunk to the CSV 
            if chunk_data:
                chunk_df = pd.DataFrame(chunk_data)
                # If file exists, append without headers. Otherwise, write new
                chunk_df.to_csv(output_csv, mode='a', header=not os.path.exists(output_csv), index=False)
            
            time.sleep(2) # Safety pause

In [41]:
build_csv_comps_table("../data/raw/company_tickers.json", "../data/raw/company_metrics.csv", 25, 50)

In [ ]:
pd.read_csv("../data/raw/company_metrics.csv")

,ticker,forwardPE,ev_to_ebitda,ebitda_margin,debt_to_ebitda,sector,industry,business_summary,ebitda,total_cash,total_debt,shares_outstanding
0,NVDA,17.739650,35.888,0.61698,0.085656,Technology,Semiconductors,NVIDIA Corporation operates as a data center s...,1.332300e+11,6.255600e+10,1.141200e+10,24300000000
1,GOOGL,25.076876,26.757,0.37279,0.446119,Communication Services,Internet Content & Information,Alphabet Inc. offers various products and plat...,1.501750e+11,1.268430e+11,6.699600e+10,5822000000
2,AAPL,28.615034,25.736,0.35100,0.591941,Technology,Consumer Electronics,"Apple Inc. designs, manufactures, and markets ...",1.529020e+11,6.690700e+10,9.050900e+10,14681140000
3,MSFT,21.752918,16.849,0.57377,0.703405,Technology,Software - Infrastructure,Microsoft Corporation develops and supports so...,1.752590e+11,8.946200e+10,1.232780e+11,7425629076
4,AMZN,26.442839,18.680,0.20327,1.225182,Consumer Cyclical,Internet Retail,"Amazon.com, Inc. engages in the retail sale of...",1.457310e+11,1.230290e+11,1.785470e+11,10754251799
5,AVGO,22.043870,51.863,0.54506,1.774867,Technology,Semiconductors,"Broadcom Inc. designs, develops, and supplies ...",3.721800e+10,1.417400e+10,6.605700e+10,4734668184
6,META,18.855963,16.707,0.50701,0.835012,Communication Services,Internet Content & Information,"Meta Platforms, Inc. engages in the developmen...",1.018920e+11,8.159200e+10,8.508100e+10,2187177748
7,TSLA,141.406310,137.340,0.11076,1.401409,Consumer Cyclical,Auto Manufacturers,"Tesla, Inc. designs, develops, manufactures, l...",1.050300e+10,4.405900e+10,1.471900e+10,3752431984
8,BRK-B,21.795288,-2.125,0.29774,1.224036,Financial Services,Insurance - Diversified,"Berkshire Hathaway Inc., together with its sub...",1.105940e+11,3.733110e+11,1.353710e+11,1390722404
9,WMT,37.981815,24.103,0.06174,1.554147,Consumer Defensive,Discount Stores,Walmart Inc. engages in the operation of retai...,4.402800e+10,1.072700e+10,6.842600e+10,7972402501


In [39]:
print(yf.Ticker("AEM").info.get("debtToEquity"))

1.299
